In [ ]:
'''
# %% Qwen 학습용 JSONL 생성 (텔롭 텍스트만)
import os
import json
import re
import polars as pl
from tqdm import tqdm

# -------------------------
# Paths / Config
# -------------------------
OCR_DIR = "./data/3_ocr_results"
TELOP_DIR = "./data/4_is_telop_results"
STT_DIR = "./data/4_stt_results"
OUTPUT_DIR = "./data"

SYSTEM_PROMPT = """당신은 유튜브 영상의 텔롭 텍스트를 생성하는 모델입니다.

입력: 채널명, 영상명, 프레임별 STT(음성 인식) 시퀀스
출력: 각 프레임에 표시될 텔롭 텍스트 (여러 개면 | 구분)

텔롭: 자막, 캡션, 이름표, 채널 로고, 타임스탬프, 리액션 텍스트 등 후편집으로 삽입된 화면 텍스트
STT는 음성이고 텔롭은 화면 텍스트이므로 동일하지 않습니다."""

# -------------------------
# 직렬화
# -------------------------
def strip_video_id(video_name: str) -> str:
    return re.sub(r"_[A-Za-z0-9_-]{11}$", "", video_name)


def serialize_stt_input(channel_name: str, video_name: str, frames: list[dict]) -> str:
    lines = [
        f"채널: {channel_name}",
        f"영상: {strip_video_id(video_name)}",
        "",
    ]
    for f in frames:
        fnum = f["frame_num"]
        stt = f["stt_text"] if f["stt_text"] is not None else "null"
        lines.append(f"[F{fnum:04d}] {stt}")
    return "\n".join(lines)


def serialize_telop_output(frames: list[dict]) -> str:
    lines = []
    for f in frames:
        fnum = f["frame_num"]
        ocr_texts = json.loads(f["ocr_texts"]) if isinstance(f["ocr_texts"], str) else f["ocr_texts"]
        is_telop = json.loads(f["is_telop"]) if isinstance(f["is_telop"], str) else f["is_telop"]

        # telop만 필터링
        telop_texts = [t for t, flag in zip(ocr_texts, is_telop) if flag is True]

        if telop_texts:
            text = " | ".join(telop_texts)
        else:
            text = "null"
        lines.append(f"[F{fnum:04d}] {text}")
    return "\n".join(lines)


def build_sample(channel_name: str, video_name: str, ocr_path: str, telop_path: str, stt_path: str) -> dict | None:
    try:
        ocr_df = pl.read_parquet(ocr_path, glob=False)
        telop_df = pl.read_parquet(telop_path, glob=False)
        stt_df = pl.read_parquet(stt_path, glob=False)
    except Exception as e:
        print(f"⚠️ 파일 읽기 실패: {e}")
        return None

    # frame_num 기준 join: ocr + telop + stt
    merged = (
        ocr_df
        .join(telop_df, on="frame_num", how="left")
        .join(stt_df, on="frame_num", how="left")
        .sort("frame_num")
    )

    frames = merged.to_dicts()
    if not frames:
        return None

    user_text = serialize_stt_input(channel_name, video_name, frames)
    assistant_text = serialize_telop_output(frames)

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text},
        ],
        "metadata": {
            "channel": channel_name,
            "video": video_name,
            "num_frames": len(frames),
        },
    }


# -------------------------
# Main
# -------------------------
def main():
    channels = sorted([
        d for d in os.listdir(OCR_DIR)
        if os.path.isdir(os.path.join(OCR_DIR, d))
    ])
    print(f"📁 총 {len(channels)}개 채널 발견")

    all_videos = []
    for channel_name in channels:
        ocr_channel_dir = os.path.join(OCR_DIR, channel_name)
        telop_channel_dir = os.path.join(TELOP_DIR, channel_name)
        stt_channel_dir = os.path.join(STT_DIR, channel_name)

        if not os.path.isdir(stt_channel_dir):
            continue
        if not os.path.isdir(telop_channel_dir):
            continue

        for pf in sorted(os.listdir(ocr_channel_dir)):
            if not pf.endswith(".parquet"):
                continue
            video_name = pf[:-8]
            ocr_path = os.path.join(ocr_channel_dir, pf)
            telop_path = os.path.join(telop_channel_dir, pf)
            stt_path = os.path.join(stt_channel_dir, pf)

            if not os.path.exists(stt_path):
                continue
            if not os.path.exists(telop_path):
                continue

            all_videos.append((channel_name, video_name, ocr_path, telop_path, stt_path))

    print(f"🎯 {len(all_videos):,}개 영상 대상")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    out_path = os.path.join(OUTPUT_DIR, "5_qwen_dataset.jsonl")
    f_out = open(out_path, "w", encoding="utf-8")

    n_saved = 0
    n_skipped = 0
    token_counts = []

    for channel_name, video_name, ocr_path, telop_path, stt_path in tqdm(all_videos, desc="샘플 생성"):
        sample = build_sample(channel_name, video_name, ocr_path, telop_path, stt_path)
        if sample is None:
            n_skipped += 1
            continue

        f_out.write(json.dumps(sample, ensure_ascii=False) + "\n")
        n_saved += 1

        total_chars = sum(len(m["content"]) for m in sample["messages"])
        token_counts.append(int(total_chars * 1.5))

    f_out.close()

    print(f"\n✅ 완료!")
    print(f"  저장: {n_saved:,}개")
    print(f"  스킵: {n_skipped:,}개")

    if token_counts:
        avg = sum(token_counts) / len(token_counts)
        mx = max(token_counts)
        print(f"  추정 토큰: 평균 {avg:,.0f}, 최대 {mx:,.0f}")

if __name__ == "__main__":
    main()
'''

📁 총 664개 채널 발견
🎯 66,400개 영상 대상


샘플 생성: 100%|██████████| 66400/66400 [08:55<00:00, 124.10it/s]


✅ 완료!
  저장: 66,400개
  스킵: 0개
  추정 토큰: 평균 45,736, 최대 838,165


In [1]:
# %% Qwen 학습용 JSONL 생성 (인스턴스 기반)
import os
import json
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# -------------------------
# Paths / Config
# -------------------------
INST_DIR = "./data/6_telop_instances"
STT_DIR  = "./data/4_stt_results"
OUT_PATH = "./data/7_qwen_dataset.jsonl"
NUM_WORKERS = 32
FPS = 10

SYSTEM_PROMPT = """당신은 유튜브 영상의 텔롭(화면 텍스트) 인스턴스를 생성하는 모델입니다.

입력 형식:
- 채널명, 영상명
- STT(음성 인식) 시퀀스: 각 줄은 `[시작초-종료초] 발화내용` 형식

출력 형식:
- JSON: {"instances": [{"text": ..., "start_sec": ..., "end_sec": ...}, ...]}
- start_sec 오름차순 정렬
- 동시에 여러 텔롭이 표시될 수 있음 (시간 겹침 허용)
- 텔롭이 없으면 빈 리스트 반환

텔롭은 자막·캡션·이름표·리액션 텍스트·로고·타임스탬프 등 후편집으로 삽입된 화면 텍스트입니다.
STT는 음성이고 텔롭은 화면 텍스트이므로 동일하지 않습니다."""


# -------------------------
# STT: frame 단위 → segment 단위 복원
# -------------------------
def stt_frames_to_segments(df_stt: pl.DataFrame) -> list[dict]:
    """
    4_stt_results는 프레임마다 해당 시점의 stt_text를 중복 저장한 구조.
    연속된 같은 stt_text를 하나의 segment로 합침.
    """
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None

    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])

        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text,
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f

    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text,
        })
    return segments


# -------------------------
# 직렬화
# -------------------------
def strip_video_id(video_name: str) -> str:
    return video_name.rsplit("__", 1)[0]


def serialize_stt_input(channel: str, video: str, segments: list[dict]) -> str:
    lines = [
        f"채널: {channel}",
        f"영상: {strip_video_id(video)}",
        "",
    ]
    for seg in segments:
        text = seg["text"].strip() if seg["text"] else ""
        if not text:
            continue
        lines.append(f"[{seg['start']:.1f}-{seg['end']:.1f}] {text}")
    return "\n".join(lines)


def serialize_instances_output(instances: list[dict]) -> str:
    """6_telop_instances와 동일 형식, compact JSON"""
    payload = {"instances": instances}
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":"))


# -------------------------
# 단일 영상 처리
# -------------------------
def build_sample(channel: str, video_name: str, inst_path: str, stt_path: str) -> dict | None:
    try:
        with open(inst_path, encoding="utf-8") as f:
            inst_data = json.load(f)
        df_stt = pl.read_parquet(stt_path, glob=False)
    except Exception:
        return None

    segments = stt_frames_to_segments(df_stt)
    instances = inst_data.get("instances", [])  # 비어있어도 그대로 []

    user_text = serialize_stt_input(channel, video_name, segments)
    assistant_text = serialize_instances_output(instances)

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant_text},
        ],
        "metadata": {
            "channel": channel,
            "video": video_name,
            "num_instances": len(instances),
            "num_stt_segments": len(segments),
        },
    }


def _worker(args):
    return build_sample(*args)


# -------------------------
# Main
# -------------------------
def main():
    tasks = []
    for channel in sorted(os.listdir(INST_DIR)):
        inst_ch = os.path.join(INST_DIR, channel)
        stt_ch  = os.path.join(STT_DIR, channel)
        if not os.path.isdir(inst_ch) or not os.path.isdir(stt_ch):
            continue
        for fname in sorted(os.listdir(inst_ch)):
            if not fname.endswith(".json"):
                continue
            video_name = fname[:-5]
            inst_path = os.path.join(inst_ch, fname)
            stt_path  = os.path.join(stt_ch, video_name + ".parquet")
            if not os.path.exists(stt_path):
                continue
            tasks.append((channel, video_name, inst_path, stt_path))

    print(f"🎯 {len(tasks):,}개 영상 대상")

    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    n_saved = n_skipped = n_empty = 0

    with open(OUT_PATH, "w", encoding="utf-8") as fout, \
         ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:

        futures = {pool.submit(_worker, t): t for t in tasks}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="샘플 생성"):
            sample = fut.result()
            if sample is None:
                n_skipped += 1
                continue
            fout.write(json.dumps(sample, ensure_ascii=False) + "\n")
            n_saved += 1
            if sample["metadata"]["num_instances"] == 0:
                n_empty += 1

    print(f"\n✅ 완료")
    print(f"  저장: {n_saved:,}개")
    print(f"  스킵: {n_skipped:,}개")
    print(f"  텔롭 없는 영상: {n_empty:,}개")


if __name__ == "__main__":
    main()

🎯 66,400개 영상 대상


샘플 생성: 100%|██████████| 66400/66400 [00:24<00:00, 2668.23it/s]



✅ 완료
  저장: 66,400개
  스킵: 0개
  텔롭 없는 영상: 6,524개
